# Laya full fine-tune on a T4

Is the ~60% plateau on the Mac a Laya limit or a Mac limit? The Mac runs trained only the top 8 of 28 layers
(and the 500-per-class pilot ran 1 epoch). Here every layer trains, in fp16, on the same complaints.

1. **Runtime → Change runtime type → T4 GPU**
2. **Runtime → Run all**; when the upload button appears, pick `laya_t4_bundle.zip`
3. Each run saves to `results/` as it finishes; the last cell downloads them (run it any time, e.g. before a disconnect).

| run | compare with (Mac) |
|---|---|
| `t4-zs` zero-shot | 48.0% — parity check, should match |
| `t4-full-n50` | 59.25% (top 8 layers, 4 epochs) |
| `t4-full-n500` | 60.15% (top 8 layers, 1 epoch); TF-IDF 73.2% |
| `t4-full-n500-rlcd` | same, with Laya's default loss instead of soft-ce |
| `t4-full-n500-1ep` | separates "more layers" from "more epochs" |


In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
!pip install -q laya==0.4.24
import torch; assert torch.cuda.is_available(), "No GPU: Runtime -> Change runtime type -> T4 GPU"

In [ ]:
import os, zipfile
if not os.path.exists("bundle/test.parquet"):
    from google.colab import files
    up = files.upload()  # laya_t4_bundle.zip
    zipfile.ZipFile(next(iter(up))).extractall("bundle")
print(sorted(os.listdir("bundle")))

In [ ]:
import gc, json, os, shutil, time
import numpy as np, pandas as pd, torch
from laya import load
from laya.train import TrainConfig, finetune

BASE = "convaiinnovations/laya"
Q = json.load(open("bundle/questions.json"))
SPLITS = {s: pd.read_parquet(f"bundle/{s}.parquet") for s in ("test", "val")}
os.makedirs("results", exist_ok=True)

def ece(conf, ok, bins=10):
    idx = np.minimum((conf * bins).astype(int), bins - 1)
    return float(sum(abs(ok[idx == b].mean() - conf[idx == b].mean()) * (idx == b).mean() for b in range(bins) if (idx == b).any()))

def run(name, train=None, **cfg):
    if os.path.exists(f"results/{name}.json"):
        return print("already done:", name)
    rec, model_dir = {"model": name, "train": train, **cfg}, BASE
    if train:
        model_dir = f"/content/ckpt/{name}"
        t0 = time.perf_counter()
        finetune(f"bundle/{train}", BASE, model_dir, TrainConfig(micro_batch=8, log_every=50, **cfg), device="cuda")
        rec["train_min"] = round((time.perf_counter() - t0) / 60, 1)
    agent = load(model_dir, device="cuda")
    for split, df in SPLITS.items():
        t0 = time.perf_counter()
        a = [agent.system_one(t, Q)["answers"]["product"] for t in df["narrative"]]
        rec[f"{split}_s_per_item"] = round((time.perf_counter() - t0) / len(df), 4)
        p = pd.DataFrame({"complaint_id": df["complaint_id"].values, "canonical_product": df["canonical_product"].values,
                          "pred": [x["choice"] for x in a], "confidence": [x["confidence"] for x in a],
                          "probabilities": [json.dumps(x["probabilities"]) for x in a]})
        p.to_parquet(f"results/{split}_{name}.parquet", index=False)
        ok = (p["pred"] == p["canonical_product"]).to_numpy()
        rec[f"{split}_acc"], rec[f"{split}_ece"] = float(ok.mean()), ece(p["confidence"].to_numpy(), ok)
    json.dump(rec, open(f"results/{name}.json", "w"))
    print(rec)
    del agent; gc.collect(); torch.cuda.empty_cache()
    if train:
        shutil.rmtree(model_dir, ignore_errors=True)  # 1.6 GB each; only predictions come back

In [ ]:
run("t4-zs")

In [ ]:
run("t4-full-n50", "train_n50.jsonl", epochs=4, grad_accum=4, loss="soft-ce")

In [ ]:
run("t4-full-n500", "train_n500.jsonl", epochs=4, grad_accum=8, loss="soft-ce")

In [ ]:
run("t4-full-n500-rlcd", "train_n500.jsonl", epochs=4, grad_accum=8, loss="rlcd")

In [ ]:
run("t4-full-n500-1ep", "train_n500.jsonl", epochs=1, grad_accum=8, loss="soft-ce")

In [ ]:
!zip -qr laya_t4_results.zip results
from google.colab import files
files.download("laya_t4_results.zip")